In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.config import (
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\pc\Desktop\yzta-2026-datathon


In [2]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [3]:
def add_features(df):
    df = df.copy()

    if {"rem_yuzdesi", "derin_uyku_yuzdesi"}.issubset(df.columns):
        df["toplam_kaliteli_uyku_yuzdesi"] = (
            df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
        )

        df["rem_derin_uyku_carpim"] = (
            df["rem_yuzdesi"] * df["derin_uyku_yuzdesi"]
        )

    if {"gecelik_uyanma_sayisi", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["uyku_bolunme_yuku"] = (
            df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
        )

        df["uyku_verimsizlik_skoru"] = (
            df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
        )

    if {"stres_skoru", "gunluk_calisma_saati"}.issubset(df.columns):
        df["stres_calisma_yuku"] = (
            df["stres_skoru"] * df["gunluk_calisma_saati"]
        )

    if {"uyku_oncesi_ekran_suresi_dk", "uyku_oncesi_kafein_mg"}.issubset(df.columns):
        df["ekran_kafein_yuku"] = (
            df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
        )

    if "gunluk_adim_sayisi" in df.columns:
        df["adim_sayisi_bin"] = df["gunluk_adim_sayisi"] / 1000

    if {"dinlenik_nabiz_bpm", "stres_skoru"}.issubset(df.columns):
        df["nabiz_stres_yuku"] = (
            df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
        )

    if "vucut_kitle_indeksi" in df.columns:
        df["bmi_kategori"] = pd.cut(
            df["vucut_kitle_indeksi"],
            bins=[0, 18.5, 25, 30, np.inf],
            labels=["zayif", "normal", "kilolu", "obez"]
        ).astype("object")

    if "gun_tipi" in df.columns:
        df["hafta_sonu_flag"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    if "ruh_sagligi_durumu" in df.columns:
        risk_map = {
            "Saglikli": 0,
            "Anksiyete": 1,
            "Depresyon": 2,
            "Anksiyete ve depresyon": 3,
        }

        df["ruh_sagligi_risk_skoru"] = df["ruh_sagligi_durumu"].map(risk_map)

    return df

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (56000, 24)
Test shape: (24000, 23)
Sample submission shape: (2, 2)


In [5]:
train_fe = add_features(train)
test_fe = add_features(test)

print("Train shape before FE:", train.shape)
print("Train shape after FE:", train_fe.shape)

print("Test shape before FE:", test.shape)
print("Test shape after FE:", test_fe.shape)

Train shape before FE: (56000, 24)
Train shape after FE: (56000, 35)
Test shape before FE: (24000, 23)
Test shape after FE: (24000, 34)


In [6]:
X = train_fe.drop(columns=[TARGET, ID_COL])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])
test_ids = test_fe[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (56000, 33)
y shape: (56000,)
X_test shape: (24000, 33)


In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

Numeric feature count: 25
Categorical feature count: 8

Categorical features:
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi', 'bmi_kategori']


In [8]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [9]:
def run_cv_model(model_name, model, X, y, X_test):
    print("=" * 80)
    print(f"Model: {model_name}")

    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), N_SPLITS))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Fold {fold}")

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        valid_pred = pipeline.predict(X_valid_fold)
        valid_pred = np.clip(valid_pred, 0, 10)

        fold_rmse = rmse(y_valid_fold, valid_pred)
        fold_scores.append(fold_rmse)

        oof_pred[valid_idx] = valid_pred

        test_pred = pipeline.predict(X_test)
        test_pred = np.clip(test_pred, 0, 10)

        test_pred_folds[:, fold - 1] = test_pred

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)

    print(f"{model_name} CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

    return {
        "model": model_name,
        "cv_rmse_mean": mean_rmse,
        "cv_rmse_std": std_rmse,
        "fold_scores": fold_scores,
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1)
    }

In [10]:
lightgbm_experiments = {
    "lightgbm_base": LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),

    "lightgbm_leaves15": LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=15,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),

    "lightgbm_leaves63": LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),

    "lightgbm_lr002": LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.02,
        max_depth=-1,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),

    "lightgbm_regularized": LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.025,
        max_depth=-1,
        num_leaves=31,
        min_child_samples=40,
        reg_alpha=0.1,
        reg_lambda=1.0,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),
}

In [11]:
xgboost_experiments = {
    "xgboost_base": XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "xgboost_depth3": XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "xgboost_depth4_lr002": XGBRegressor(
        n_estimators=5000,
        learning_rate=0.02,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "xgboost_regularized": XGBRegressor(
        n_estimators=4000,
        learning_rate=0.025,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

In [12]:
all_results = []
all_oof_predictions = {}
all_test_predictions = {}

all_experiments = {}
all_experiments.update(lightgbm_experiments)
all_experiments.update(xgboost_experiments)

for model_name, model in all_experiments.items():
    result = run_cv_model(model_name, model, X, y, X_test)

    all_results.append({
        "model": result["model"],
        "cv_rmse_mean": result["cv_rmse_mean"],
        "cv_rmse_std": result["cv_rmse_std"],
        "fold_scores": result["fold_scores"]
    })

    all_oof_predictions[model_name] = result["oof_pred"]
    all_test_predictions[model_name] = result["test_pred"]

Model: lightgbm_base
Fold 1


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 RMSE: 1.23984
Fold 2


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 RMSE: 1.24165
Fold 3


...

Fold 3 RMSE: 1.22975
Fold 4


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 RMSE: 1.23626
Fold 5


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 RMSE: 1.25667
lightgbm_regularized CV RMSE: 1.24108 ± 0.00894
Model: xgboost_base
Fold 1
Fold 1 RMSE: 1.24178
Fold 2
Fold 2 RMSE: 1.24173
Fold 3
Fold 3 RMSE: 1.22645
Fold 4
Fold 4 RMSE: 1.24013
Fold 5
Fold 5 RMSE: 1.25726
xgboost_base CV RMSE: 1.24147 ± 0.00977
Model: xgboost_depth3
Fold 1
Fold 1 RMSE: 1.22897
Fold 2
Fold 2 RMSE: 1.22703
Fold 3
Fold 3 RMSE: 1.21280
Fold 4
Fold 4 RMSE: 1.22266
Fold 5
Fold 5 RMSE: 1.24125
xgboost_depth3 CV RMSE: 1.22654 ± 0.00923
Model: xgboost_depth4_lr002
Fold 1
Fold 1 RMSE: 1.23337
Fold 2
Fold 2 RMSE: 1.23425
Fold 3
Fold 3 RMSE: 1.21936
Fold 4
Fold 4 RMSE: 1.23207
Fold 5
Fold 5 RMSE: 1.24937
xgboost_depth4_lr002 CV RMSE: 1.23368 ± 0.00953
Model: xgboost_regularized
Fold 1
Fold 1 RMSE: 1.23364
Fold 2
Fold 2 RMSE: 1.23423
Fold 3
Fold 3 RMSE: 1.21964
Fold 4
Fold 4 RMSE: 1.23138
Fold 5
Fold 5 RMSE: 1.24892
xgboost_regularized CV RMSE: 1.23356 ± 0.00933


In [13]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("cv_rmse_mean").reset_index(drop=True)

results_df[["model", "cv_rmse_mean", "cv_rmse_std"]]

,model,cv_rmse_mean,cv_rmse_std
0,xgboost_depth3,1.226544,0.009234
1,lightgbm_leaves15,1.231572,0.009623
2,xgboost_regularized,1.233563,0.009328
3,xgboost_depth4_lr002,1.233682,0.009531
4,lightgbm_lr002,1.239837,0.009146
5,lightgbm_base,1.240055,0.009308
6,lightgbm_regularized,1.241077,0.008944
7,xgboost_base,1.241470,0.009766
8,lightgbm_leaves63,1.246518,0.008671


In [14]:
best_model_name = results_df.loc[0, "model"]
best_test_pred = all_test_predictions[best_model_name]

best_test_pred = np.clip(best_test_pred, 0, 10)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_test_pred
})

print("Best model:", best_model_name)
print("Submission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

display(submission.head())

Best model: xgboost_depth3
Submission shape: (24000, 2)
Submission columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.981785
1,2,6.394166
2,3,2.842898
3,4,7.103512
4,5,3.745434


In [15]:
print("Our submission shape:", submission.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nOur columns:")
print(submission.columns.tolist())

print("\nSample columns:")
print(sample_submission.columns.tolist())

display(submission.head())
display(sample_submission.head())

Our submission shape: (24000, 2)
Sample submission shape: (2, 2)

Our columns:
['id', 'bilissel_performans_skoru']

Sample columns:
['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.981785
1,2,6.394166
2,3,2.842898
3,4,7.103512
4,5,3.745434


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32


In [16]:
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

submission_path = SUBMISSION_DIR / f"submission_{best_model_name}.csv"

submission.to_csv(submission_path, index=False)

print("Best model:", best_model_name)
print("Saved:", submission_path)
print("Shape:", submission.shape)

Best model: xgboost_depth3
Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_xgboost_depth3.csv
Shape: (24000, 2)


In [17]:
check_submission = pd.read_csv(submission_path)

print("Check shape:", check_submission.shape)
print("Check columns:", check_submission.columns.tolist())

display(check_submission.head())

Check shape: (24000, 2)
Check columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.981785
1,2,6.394166
2,3,2.842898
3,4,7.103512
4,5,3.745434


In [18]:
PREDICTION_DIR = PROJECT_ROOT / "predictions" / "lgbm_xgb"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for model_name in all_oof_predictions.keys():
    oof_df = pd.DataFrame({
        ID_COL: train[ID_COL],
        "y_true": y,
        "oof_pred": np.clip(all_oof_predictions[model_name], 0, 10)
    })

    test_df = pd.DataFrame({
        ID_COL: test_ids,
        "test_pred": np.clip(all_test_predictions[model_name], 0, 10)
    })

    oof_path = PREDICTION_DIR / f"{model_name}_oof.csv"
    test_path = PREDICTION_DIR / f"{model_name}_test.csv"

    oof_df.to_csv(oof_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Saved:", oof_path)
    print("Saved:", test_path)

Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_base_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_base_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_leaves15_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_leaves15_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_leaves63_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_leaves63_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_lr002_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_lr002_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_regularized_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\lightgbm_regularized_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\lgbm_xgb\xgboos